# Diet Model Evaluation: Metrics + Visualizations

Loads the **real saved model and datasets** from this repo — nothing is hardcoded.
Computes precision, recall, F1 (macro/weighted), confusion matrix, feature importance,
and food-catalog stats, with plots.

**In Colab:** upload this repo (or `!git clone <your-repo-url>`), then set `REPO` below
to the repo root, e.g. `REPO = "/content/<repo-name>"`. Locally, just run as-is.

In [ ]:
from pathlib import Path

def _find_repo():
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "artifacts" / "diet_model.joblib").exists():
            return p
    return Path("/content/repo")

REPO = _find_repo()
assert (REPO / "artifacts" / "diet_model.joblib").exists(), f"artifacts not found under {REPO}"
print("repo:", REPO)

In [ ]:
# !pip install -q scikit-learn pandas matplotlib seaborn  # uncomment if imports fail

import json

import joblib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.metrics import (accuracy_score, classification_report,
                               confusion_matrix, precision_recall_fscore_support)

sns.set_theme()

## 1. Load data, model, and saved train metadata

In [ ]:
from sklearn.model_selection import train_test_split

df = None  # loaded + prepared in section 2
model = joblib.load(REPO / "artifacts" / "diet_model.joblib")
meta = json.loads((REPO / "artifacts" / "diet_meta.json").read_text())
print("target:", meta["target"], "| classes:", sorted(meta["target_distribution"]))
print("target distribution:", meta["target_distribution"])
print("excluded from features:", meta["excluded"])
print("features:", len(meta["features_numeric"]), "numeric +",
      len(meta["features_categorical"]), "categorical")

## 2. Reproduce the held-out evaluation (same 80/20 split, seed 42)

This re-runs the model on the held-out test split — it does **not** evaluate on training data.

In [ ]:
import sys

sys.path.insert(0, str(REPO))
from ml.data_preparation import diet_features, prepare_diet_frame  # same prep as training

# NOTE: raw CSV must go through prepare_diet_frame first (NaN -> 'None' levels),
# exactly as in training — otherwise test inputs differ from train inputs.
df = prepare_diet_frame(REPO / "datasets" / "diet_recommendations_dataset.csv")
X, y = diet_features(df)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42,
                                        stratify=y)
pred = model.predict(Xte)
labels = sorted(y.unique())
print("test size:", len(yte))
print("accuracy:", round(accuracy_score(yte, pred), 4))

## 3. Precision, recall, F1 per class

In [ ]:
report = classification_report(yte, pred, zero_division=0, output_dict=True)
per_class = pd.DataFrame(
    {k: {"precision": v["precision"], "recall": v["recall"],
            "f1-score": v["f1-score"], "support": v["support"]}
     for k, v in report.items() if k in labels}).T
print(per_class.round(4).to_string())
print()
for avg in ["macro avg", "weighted avg"]:
    v = report[avg]
    print(f"{avg}: precision={v['precision']:.4f} recall={v['recall']:.4f} "
          f"f1={v['f1-score']:.4f}")

per_class[["precision", "recall", "f1-score"]].plot.bar(figsize=(8, 4))
plt.title("Per-class precision / recall / F1 (held-out test set)")
plt.ylabel("score")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 4. Confusion matrix

In [ ]:
cm = confusion_matrix(yte, pred, labels=labels)
cm_df = pd.DataFrame(cm, index=[f"true {l}" for l in labels],
                     columns=[f"pred {l}" for l in labels])
print(cm_df.to_string())

plt.figure(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
plt.title("Confusion matrix — diet RandomForest (test set)")
plt.tight_layout()
plt.show()

## 5. Target distribution vs predicted distribution

In [ ]:
dist = pd.DataFrame({
    "train_target": pd.Series(meta["target_distribution"]),
    "test_true": yte.value_counts(),
    "test_predicted": pd.Series(pred).value_counts(),
})
print(dist.to_string())
dist.plot.bar(figsize=(8, 4))
plt.title("Class balance: train vs test-true vs test-predicted")
plt.ylabel("count")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 6. Feature importance (from the trained forest, aggregated to base columns)

In [ ]:
imp = pd.DataFrame(meta["feature_importance"]).set_index("feature")["importance"]
print("sums to:", round(float(imp.sum()), 6))
print(imp.sort_values(ascending=False).round(4).to_string())

imp.sort_values().plot.barh(figsize=(8, 6))
plt.title("RandomForest feature importance (aggregated from one-hot)")
plt.xlabel("importance")
plt.tight_layout()
plt.show()

## 7. Model vs dummy baseline (saved metrics)

In [ ]:
saved = json.loads((REPO / "reports" / "metrics" / "diet_metrics.json").read_text())
comp = pd.DataFrame({
    "random_forest": {k: saved["random_forest"][k]
                      for k in ["accuracy", "precision_macro", "recall_macro",
                                    "f1_macro", "f1_weighted"]},
    "dummy_stratified": {k: saved["dummy_stratified"][k]
                         for k in ["accuracy", "precision_macro", "recall_macro",
                                       "f1_macro", "f1_weighted"]},
})
print(comp.round(4).to_string())
comp.plot.bar(figsize=(8, 4))
plt.title("RandomForest vs stratified dummy (same split)")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()
print("\nTakeaway: the forest barely beats dummy — weak signal, informational use only.")

## 8. KNN food-catalog nutrition overview

In [ ]:
cat = pd.read_csv(REPO / "artifacts" / "indian_catalog.csv")
cols = ["energy_kcal", "proteins_100g", "fat_100g", "carbs_100g",
        "sugars_100g", "fiber_100g", "sodium_100g"]
print("rows:", len(cat))
print("missing %:", (cat[cols].isna().mean() * 100).round(1).to_dict())
cat[cols].hist(figsize=(12, 6), bins=30)
plt.suptitle("Indian catalog nutrition distributions (per 100 g)")
plt.tight_layout()
plt.show()